<a href="https://colab.research.google.com/github/huwperkins08-lang/TSF_seepage_detection_ElSoldado/blob/main/REP_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap.core as geemap

# 1. INITIALIZATION
ee.Authenticate()
ee.Initialize(project='sxe390-el-soldado')

# 2. PARAMETERS & GEOMETRY
lat, lon = -32.65, -71.16
mine_site = ee.Geometry.Point([lon, lat])
study_area = mine_site.buffer(20000)

# 3. HELPER FUNCTIONS
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

def resample_to_20m(image):
    bands_10m = ['B2', 'B3', 'B4', 'B8']
    bands_20m = ['B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
    target_projection = image.select('B5').projection()
    downsampled = (image.select(bands_10m)
                   .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
                   .reproject(crs=target_projection))
    return image.addBands(downsampled, overwrite=True).select(bands_10m + bands_20m)

def apply_terrain_mask(image):
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem)
    return image.updateMask(slope.lte(20)) # Mask slopes > 20 degrees

def apply_illumination_mask(image):
    # 1. Safely get Sun angles. If they are missing, 'zen_val' becomes None
    zen_val = image.get('MEAN_SOLAR_ZENITH_ANGLE')
    azi_val = image.get('MEAN_SOLAR_AZIMUTH_ANGLE')

    # 2. Use a conditional: If the metadata exists, do the math. If not, return the original image.
    # This prevents the 'null' multiplication error.
    return ee.Image(ee.Algorithms.If(
        zen_val, # The condition: 'Does this value exist?'
        # TRUE CASE: Do the illumination math
        (lambda: (
            image.updateMask(
                # Math moved inside here to stay safe
                (terrain_math(image, zen_val, azi_val))
            )
        ))(),
        # FALSE CASE: Just return the image without the illumination mask
        image
    ))

# This is a helper function to keep the logic clean inside the 'If' statement
def terrain_math(image, zen, azi):
    degree2rad = 3.14159 / 180.0
    z = ee.Number(zen).multiply(degree2rad)
    a = ee.Number(azi).multiply(degree2rad)

    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    terrain = ee.Terrain.products(dem)
    s = terrain.select('slope').multiply(degree2rad)
    asp = terrain.select('aspect').multiply(degree2rad)

    term1 = s.cos().multiply(z.cos())
    term2 = s.sin().multiply(z.sin()).multiply(asp.subtract(a).cos())
    cos_i = term1.add(term2)
    return cos_i.gt(0.1)

def calculate_rep(image):
    b4, b5, b6, b7 = image.select('B4'), image.select('B5'), image.select('B6'), image.select('B7')
    re_reflectance = (b4.add(b7)).divide(2)
    rep = ee.Image(705).add(
        ee.Image(35).multiply(
            re_reflectance.subtract(b5).divide(b6.subtract(b5))
        )
    ).rename('REP')
    ndvi = image.normalizedDifference(['B8', 'B4'])
    return image.addBands(rep).updateMask(ndvi.gt(0.3))

# 4. SOURCE & FILTER
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(mine_site)
                  .filterDate('2023-01-01', '2023-12-31')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
                  .filter(ee.Filter.notNull(['MEAN_SOLAR_ZENITH_ANGLE', 'MEAN_SOLAR_AZIMUTH_ANGLE'])))

# 5. EXECUTION PIPELINE
processed_collection = (s2_collection
                        .map(mask_s2_clouds)
                        .map(resample_to_20m)
                        .map(apply_terrain_mask)
                        .map(apply_illumination_mask) # <--- Added here
                        .map(calculate_rep))

# 6. GENERATE OUTPUT
final_median = processed_collection.median().clip(study_area)

# 7. VISUALIZATION
Map = geemap.Map(center=[lat, lon], zoom=13)
Map.addLayer(final_median, {'bands':['B4','B3','B2'], 'min':0, 'max':0.25}, 'Natural Color')

rep_params = {'bands':['REP'], 'min': 705, 'max': 720, 'palette': ['red', 'orange', 'yellow', 'green']}
Map.addLayer(final_median, rep_params, 'Red Edge Position (REP)')

Map

In [ ]:
import geopandas as gpd
import ee
import geemap # Use top-level geemap for ee_to_df

# 1. Load  QGIS GeoJSON
gdf = gpd.read_file('sites_final_300m.geojson')

# 2. Convert to Earth Engine FeatureCollection
def geodf_to_ee(gdf):
    features = []
    for _, row in gdf.iterrows():
        # Convert QGIS geometry to EE geometry using the full GeoJSON interface
        # This handles various geometry types (Polygon, MultiPolygon, etc.) robustly.
        ee_geometry = ee.Geometry(row.geometry.__geo_interface__)
        f = ee.Feature(ee_geometry, row.drop('geometry').to_dict())
        features.append(f)
    return ee.FeatureCollection(features)

study_sites = geodf_to_ee(gdf)

# 3. Extract EVERY pixel value (REP) for each site
# This creates a table where each row is a 10m pixel
pixel_data = rep_cr.sampleRegions(
    collection=study_sites,
    properties=['site_type'], # This links the pixel to your QGIS label
    scale=10,
    geometries=True # Keeps the lat/long of each pixel if needed
)

# 4. Convert the massive amount of data to a Pandas DataFrame
# Use geemap.ee_to_df for direct conversion of FeatureCollection to DataFrame
# Ensure 'site_name' is explicitly selected as a property for conversion
df_pixels = geemap.ee_to_df(pixel_data.select(['REP_CR','site_type']))

print(f"Successfully extracted {len(df_pixels)} individual pixel values!")
print(df_pixels.head())

In [ ]:
# This will show exactly how many pixels were found for each site type
print("Breakdown of pixels by Site Type:")
print(df_pixels['site_type'].value_counts())

# This will show the average REP value for Test vs Control
print("\nMean REP values:")
print(df_pixels.groupby('site_type')['REP_CR'].mean())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# 1. Prepare the Data - we ensure 'Test' is the first color (Purple)
site_names = ['Test', 'Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5']
colors = sns.color_palette('viridis', n_colors=len(site_names))

plt.figure(figsize=(12, 7))

# 2. Draw the curves - we use the exact same order here
sns.kdeplot(data=df_pixels, x='REP_CR', hue='site_type', hue_order=site_names,
            fill=True, common_norm=False, palette='viridis', alpha=0.3)

# 3. Add the Mean Line
test_mean = df_pixels[df_pixels['site_type'] == 'Test']['REP_CR'].mean()
mean_line = plt.axvline(test_mean, color='red', linestyle='--', label=f'Test Mean: {test_mean:.2f}')

# 4. Create the Legend handles manually to match the order
legend_handles = []
for i, name in enumerate(site_names):
    patch = mpatches.Patch(color=colors[i], label=name, alpha=0.5)
    legend_handles.append(patch)
legend_handles.append(mean_line)

# 5. Final Plot Touches
plt.legend(handles=legend_handles, title="Study Sites", loc='upper left', frameon=True)
plt.title('Red Edge Position (REP) Distribution: El Soldado Analysis', fontsize=16)
plt.xlabel('Red Edge Position (nm)', fontsize=13)
plt.ylabel('Pixel Density', fontsize=13)

plt.show()

In [ ]:
# 1. Create a clean filename
filename = 'El_Soldado_REP_Raw_Data_23_24.csv'

# 2. Export the entire DataFrame to CSV
# This includes every single pixel's value and its site name
df_pixels.to_csv(filename, index=False)

print(f"Success! '{filename}' has been created.")
print("Look in the folder icon on the left of your screen to download it.")

In [ ]:
import scipy.stats as stats

# 1. Group the data by site
groups = [group['REP_CR'].values for name, group in df_pixels.groupby('site_type')]

# 2. Perform One-Way ANOVA
f_stat, p_value = stats.f_oneway(*groups)

print("--- ANOVA RESULTS ---")
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value:     {p_value:.10f}")

# 3. Quick interpretation
if p_value < 0.05:
    print("\nResult: STATISTICALLY SIGNIFICANT")
    print("There is a significant difference in the Red Edge Position between your sites.")
else:
    print("\nResult: NOT SIGNIFICANT")

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Perform Tukey's Honestly Significant Difference (HSD) test
tukey = pairwise_tukeyhsd(endog=df_pixels['REP_CR'],
                          groups=df_pixels['site_type'],
                          alpha=0.05)

# 2. Print the results table
print("--- TUKEY POST-HOC TEST RESULTS ---")
print(tukey)

# 3. Optional: Export this table to a CSV for your report
import pandas as pd
tukey_data = pd.DataFrame(data=tukey._results_table.data[1:],
                          columns=tukey._results_table.data[0])
tukey_data.to_csv('tukey_results_elsoldado.csv', index=False)